# Phase 0: Task A Augmentation Pipeline

**Muc tieu**: Augmentation offline cho 2 lop thieu so (Podocarpaceae, Taxaceae) trong Task A.

Pipeline sinh anh augmented duoi dang file thuc tai `data/task-A/`, dua moi lop thieu so
len 1,500-1,700 mau (tuong duong muc Pinaceae: 1,666).

**Phase**: 0 (data preparation, chay local trong conda env `wood`).
Buoc upload o cuoi notebook tao artifact dataset cho cac phase sau (Colab training).

---

## Contents

1. [Root Detection](#1-setup)
2. [Verify Source Data](#2-verify-source)
3. [Import & Config](#3-import-config)
4. [Directory Setup & Copy Engine](#4-copy-engine)
5. [Augmentation Pipeline Execution](#5-augmentation)
6. [Manifest CSV Generation](#6-manifest)
7. [QA Summary](#7-qa-summary)
8. [Upload to HuggingFace](#8-upload)

## 1. Root Detection <a id='1-setup'></a>

Phase 0 chay local trong conda env `wood`. ROOT duoc phat hien tu vi tri notebook
bang cach di nguoc len cay thu muc cho den khi tim thay marker (`.kiro/` hoac `requirements.txt`).

Gia su conda env `wood` da duoc activate va cac dependencies da duoc cai dat
(albumentations, pillow, huggingface_hub, pandas, numpy).

In [1]:
from pathlib import Path
import sys

# Tim repo root bang cach di len cho den khi gap marker `.kiro/` hoac `requirements.txt`.
# Notebook nam tai `notebooks/phase 0/3_task_a_augmentation.ipynb`, nen cwd thuong la
# `<repo>/notebooks/phase 0`.
def _find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / ".kiro").is_dir() or (candidate / "requirements.txt").exists():
            return candidate
    # Fallback: gia dinh cay thu muc chuan `notebooks/phase 0/`
    return start.parent.parent

ROOT = _find_repo_root(Path.cwd())
print(f"[Local] ROOT = {ROOT}")

# Them repo root vao sys.path de `from src.data import ...` hoat dong
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f"sys.path[0] = {sys.path[0]}")

[Local] ROOT = /home/sakana/Code/ManifoldsTeam/woodbench
sys.path[0] = /home/sakana/Code/ManifoldsTeam/woodbench


## 2. Verify Source Data <a id='2-verify-source'></a>

Phase 0 yeu cau du lieu nguon da co san o local tai `data/split/` (ket qua cua notebook
`data preparation.ipynb`). Cell duoi day kiem tra su ton tai cua thu muc va liet ke
cac splits + families.

In [2]:
# Sanity-check: du lieu nguon phai ton tai tai data/split/
SRC_DIR = ROOT / "data" / "split"

if not SRC_DIR.exists():
    raise FileNotFoundError(
        f"Du lieu nguon khong ton tai tai {SRC_DIR}. "
        "Phase 0 yeu cau du lieu da duoc chuan bi truoc o local. "
        "Hay chay notebook `notebooks/phase 0/data preparation.ipynb` "
        "truoc khi chay notebook nay."
    )

print(f"SRC_DIR = {SRC_DIR}")
print("\n--- Cau truc du lieu nguon ---")
for split in ["train", "val", "test"]:
    split_path = SRC_DIR / split
    if not split_path.exists():
        raise FileNotFoundError(
            f"Thieu split: {split_path}. Du lieu nguon khong day du."
        )
    families = sorted(d.name for d in split_path.iterdir() if d.is_dir())
    print(f"  {split}/: {len(families)} families - {families}")

SRC_DIR = /home/sakana/Code/ManifoldsTeam/woodbench/data/split

--- Cau truc du lieu nguon ---
  train/: 4 families - ['Cupressaceae', 'Pinaceae', 'Podocarpaceae', 'Taxaceae']
  val/: 4 families - ['Cupressaceae', 'Pinaceae', 'Podocarpaceae', 'Taxaceae']
  test/: 4 families - ['Cupressaceae', 'Pinaceae', 'Podocarpaceae', 'Taxaceae']


## 3. Import & Config <a id='3-import-config'></a>

Import cac module augmentation va dinh nghia cau hinh pipeline.

In [3]:
from src.data.augmentation import AugmentationPipeline, AugmentationMethod
from src.data.directory_copier import DirectoryCopier
from src.data.manifest_writer import ManifestWriter

import numpy as np
from PIL import Image

# --- Cau hinh pipeline ---
SEED = 42
TARGET_COUNT = 1600  # Muc tieu so mau cho moi lop thieu so

# Cac family trong dataset
MINORITY_FAMILIES = ["Podocarpaceae", "Taxaceae"]
MAJORITY_FAMILIES = ["Cupressaceae", "Pinaceae"]
ALL_FAMILIES = MAJORITY_FAMILIES + MINORITY_FAMILIES

# Duong dan dua tren ROOT (SRC_DIR da duoc dinh nghia trong cell verify source data)
DST_DIR = ROOT / "data" / "task-A"

print(f"SEED = {SEED}")
print(f"TARGET_COUNT = {TARGET_COUNT}")
print(f"MINORITY_FAMILIES = {MINORITY_FAMILIES}")
print(f"MAJORITY_FAMILIES = {MAJORITY_FAMILIES}")
print(f"SRC_DIR = {SRC_DIR}")
print(f"DST_DIR = {DST_DIR}")

/home/sakana/miniconda3/envs/wood/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SEED = 42
TARGET_COUNT = 1600
MINORITY_FAMILIES = ['Podocarpaceae', 'Taxaceae']
MAJORITY_FAMILIES = ['Cupressaceae', 'Pinaceae']
SRC_DIR = /home/sakana/Code/ManifoldsTeam/woodbench/data/split
DST_DIR = /home/sakana/Code/ManifoldsTeam/woodbench/data/task-A


## 4. Directory Setup & Copy Engine <a id='4-copy-engine'></a>

Tao cau truc thu muc `data/task-A/` va copy anh goc:
- Val/Test: copy nguyen ban (tat ca families)
- Train majority: copy nguyen ban
- Train minority: copy nguyen ban (augmentation o buoc sau)

In [4]:
copier = DirectoryCopier()
total_copied = 0

# Copy val va test splits nguyen ban (tat ca families)
for split in ['val', 'test']:
    count = copier.copy_split(SRC_DIR, DST_DIR, split, ALL_FAMILIES)
    total_copied += count
    print(f"  {split}: {count} files copied")

# Copy train majority classes nguyen ban
count_majority = copier.copy_split(SRC_DIR, DST_DIR, 'train', MAJORITY_FAMILIES)
total_copied += count_majority
print(f"  train (majority): {count_majority} files copied")

# Copy train minority classes originals
count_minority = copier.copy_split(SRC_DIR, DST_DIR, 'train', MINORITY_FAMILIES)
total_copied += count_minority
print(f"  train (minority originals): {count_minority} files copied")

print(f"\nTong cong: {total_copied} files copied vao {DST_DIR}")

  val: 955 files copied
  test: 981 files copied
  train (majority): 4064 files copied
  train (minority originals): 423 files copied

Tong cong: 6423 files copied vao /home/sakana/Code/ManifoldsTeam/woodbench/data/task-A


## 5. Augmentation Pipeline Execution <a id='5-augmentation'></a>

Sinh anh augmented cho cac lop thieu so (Podocarpaceae, Taxaceae).
Su dung 7 methods: HFlip, VFlip, Rot180, Rot270, CropResize, ColorJitter, Transpose.
Phan phoi deu methods theo round-robin.

In [5]:
# Khoi tao pipeline
pipeline = AugmentationPipeline(seed=SEED)

# Khoi tao manifest writer
manifest_path = DST_DIR / "augmentation_manifest.csv"
manifest = ManifestWriter(manifest_path)

# Thuc hien augmentation cho tung minority family
augmentation_stats = {}  # Luu thong ke augmentation

for family in MINORITY_FAMILIES:
    print(f"\n{'='*60}")
    print(f"Augmenting: {family}")
    print(f"{'='*60}")
    
    # Tim tat ca species trong family
    family_dir = DST_DIR / "train" / family
    if not family_dir.exists():
        print(f"  WARN: {family_dir} khong ton tai, bo qua.")
        continue
    
    family_augmented = 0
    
    for species_dir in sorted(family_dir.iterdir()):
        if not species_dir.is_dir():
            continue
        
        species = species_dir.name
        
        # Lay danh sach anh goc
        original_images = sorted([
            f for f in species_dir.iterdir()
            if f.is_file() and f.suffix.lower() in {'.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp'}
        ])
        original_count = len(original_images)
        print(f"  {species}: {original_count} originals")
        
        # Tinh toan augmentation plan
        plan = pipeline.compute_augmentation_plan(original_count, TARGET_COUNT)
        print(f"    -> Can sinh them: {len(plan)} augmented images")
        
        # Thuc hien augmentation
        for idx, (img_idx, method) in enumerate(plan):
            src_path = original_images[img_idx]
            
            # Doc anh goc
            img = np.array(Image.open(src_path))
            
            # Ap dung augmentation
            augmented = pipeline.augment_image(img, method)
            
            # Tao ten file augmented: {original_stem}_aug{method}_{idx}.png
            aug_filename = f"{src_path.stem}_aug{method.value}_{idx}.png"
            aug_path = species_dir / aug_filename
            
            # Luu anh augmented
            Image.fromarray(augmented).save(aug_path)
            
            # Ghi vao manifest
            # Duong dan tuong doi tu DST_DIR
            original_rel = str(src_path.relative_to(DST_DIR))
            augmented_rel = str(aug_path.relative_to(DST_DIR))
            manifest.add_entry(
                original_path=original_rel,
                augmented_path=augmented_rel,
                method=method.value,
                family=family,
                species=species,
            )
            
            family_augmented += 1
            
            # Hien thi progress moi 200 anh
            if (idx + 1) % 200 == 0:
                print(f"    Progress: {idx + 1}/{len(plan)}")
        
        print(f"    Hoan tat: {original_count} + {len(plan)} = {original_count + len(plan)} total")
    
    augmentation_stats[family] = {
        'augmented': family_augmented,
    }
    print(f"  Tong augmented cho {family}: {family_augmented}")

print(f"\nAugmentation hoan tat.")


Augmenting: Podocarpaceae
  Podocarpus pilgeri: 217 originals
    -> Can sinh them: 1383 augmented images
    Progress: 200/1383
    Progress: 400/1383
    Progress: 600/1383
    Progress: 800/1383
    Progress: 1000/1383
    Progress: 1200/1383
    Hoan tat: 217 + 1383 = 1600 total
  Tong augmented cho Podocarpaceae: 1383

Augmenting: Taxaceae
  Taxus wallichiana: 206 originals
    -> Can sinh them: 1394 augmented images
    Progress: 200/1394
    Progress: 400/1394
    Progress: 600/1394
    Progress: 800/1394
    Progress: 1000/1394
    Progress: 1200/1394
    Hoan tat: 206 + 1394 = 1600 total
  Tong augmented cho Taxaceae: 1394

Augmentation hoan tat.


## 6. Manifest CSV Generation <a id='6-manifest'></a>

Ghi manifest CSV theo doi provenance cua tat ca file augmented.

In [6]:
# Luu manifest CSV
saved_path = manifest.save()
print(f"Manifest CSV da luu tai: {saved_path}")

# Hien thi vai dong dau
import pandas as pd
df_manifest = pd.read_csv(saved_path)
print(f"\nTong so entries: {len(df_manifest)}")
print(f"\nCac cot: {list(df_manifest.columns)}")
print(f"\n5 dong dau:")
df_manifest.head()

Manifest CSV da luu tai: /home/sakana/Code/ManifoldsTeam/woodbench/data/task-A/augmentation_manifest.csv

Tong so entries: 2777

Cac cot: ['original_path', 'augmented_path', 'method', 'family', 'species']

5 dong dau:


,original_path,augmented_path,method,family,species
0,train/Podocarpaceae/Podocarpus pilgeri/3386. P...,train/Podocarpaceae/Podocarpus pilgeri/3386. P...,hflip,Podocarpaceae,Podocarpus pilgeri
1,train/Podocarpaceae/Podocarpus pilgeri/3386. P...,train/Podocarpaceae/Podocarpus pilgeri/3386. P...,vflip,Podocarpaceae,Podocarpus pilgeri
2,train/Podocarpaceae/Podocarpus pilgeri/3386. P...,train/Podocarpaceae/Podocarpus pilgeri/3386. P...,rot180,Podocarpaceae,Podocarpus pilgeri
3,train/Podocarpaceae/Podocarpus pilgeri/3386. P...,train/Podocarpaceae/Podocarpus pilgeri/3386. P...,rot270,Podocarpaceae,Podocarpus pilgeri
4,train/Podocarpaceae/Podocarpus pilgeri/3386. P...,train/Podocarpaceae/Podocarpus pilgeri/3386. P...,crop_resize,Podocarpaceae,Podocarpus pilgeri


## 7. QA Summary <a id='7-qa-summary'></a>

Hien thi bang tom tat: so luong mau truoc/sau augmentation, phan phoi method, ti le can bang lop.

In [7]:
import pandas as pd

# --- Bang so luong mau truoc/sau augmentation ---
print("=" * 60)
print("SUMMARY: So luong mau train split truoc/sau augmentation")
print("=" * 60)

summary_data = []
for family in ALL_FAMILIES:
    family_dir = DST_DIR / "train" / family
    if not family_dir.exists():
        continue
    
    # Dem tat ca file anh trong family
    total_count = 0
    for species_dir in family_dir.iterdir():
        if species_dir.is_dir():
            total_count += len([
                f for f in species_dir.iterdir()
                if f.is_file() and f.suffix.lower() in {'.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp'}
            ])
    
    # So luong goc (tu source)
    src_family_dir = SRC_DIR / "train" / family
    original_count = 0
    if src_family_dir.exists():
        for species_dir in src_family_dir.iterdir():
            if species_dir.is_dir():
                original_count += len([
                    f for f in species_dir.iterdir()
                    if f.is_file() and f.suffix.lower() in {'.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp'}
                ])
    
    augmented_count = total_count - original_count
    summary_data.append({
        'Family': family,
        'Original': original_count,
        'Augmented': augmented_count,
        'Total': total_count,
    })

df_summary = pd.DataFrame(summary_data)
print(df_summary.to_string(index=False))

# --- Phan phoi method ---
print(f"\n{'='*60}")
print("Phan phoi augmentation methods")
print("=" * 60)

df_manifest = pd.read_csv(DST_DIR / "augmentation_manifest.csv")
method_dist = df_manifest.groupby(['family', 'method']).size().unstack(fill_value=0)
print(method_dist.to_string())

# --- Ti le can bang lop ---
print(f"\n{'='*60}")
print("Ti le can bang lop (train split)")
print("=" * 60)

total_train = df_summary['Total'].sum()
for _, row in df_summary.iterrows():
    ratio = row['Total'] / total_train * 100
    print(f"  {row['Family']}: {row['Total']} ({ratio:.1f}%)")

print(f"\n  Tong train samples: {total_train}")

SUMMARY: So luong mau train split truoc/sau augmentation
       Family  Original  Augmented  Total
 Cupressaceae      2398          0   2398
     Pinaceae      1666          0   1666
Podocarpaceae       217       1383   1600
     Taxaceae       206       1394   1600

Phan phoi augmentation methods
method         color_jitter  crop_resize  hflip  rot180  rot270  transpose  vflip
family                                                                           
Podocarpaceae           197          197    198     198     198        197    198
Taxaceae                199          199    200     199     199        199    199

Ti le can bang lop (train split)
  Cupressaceae: 2398 (33.0%)
  Pinaceae: 1666 (22.9%)
  Podocarpaceae: 1600 (22.0%)
  Taxaceae: 1600 (22.0%)

  Tong train samples: 7264


## 8. Upload to HuggingFace <a id='8-upload'></a>

Day la buoc cuoi cua phase 0 — upload toan bo `data/task-A/` len HuggingFace Hub
de tao dataset artifact cho cac phase sau (Colab training se pull artifact nay).
Yeu cau: da chay `huggingface-cli login` hoac set bien moi truong `HF_TOKEN`.

In [11]:
# Upload artifact phase 0 len HuggingFace Hub.
# Cac phase sau (Colab training cho Task A) se pull dataset nay tu HuggingFace.
from huggingface_hub import upload_folder
import os
import dotenv

dotenv.load_dotenv()

REPO_ID = "ny0g507h3p/wood-task-A"

print(f"Uploading {DST_DIR} len {REPO_ID}...")
print(f"Luu y: Qua trinh upload co the mat vai phut tuy kich thuoc dataset.")

try:
    upload_folder(
        folder_path=str(DST_DIR),
        repo_id=REPO_ID,
        repo_type="dataset",
    )
    print(f"\nUpload thanh cong!")
    print(f"Repository URL: https://huggingface.co/datasets/{REPO_ID}")

except Exception as e:
    error_msg = str(e)
    if 'authentication' in error_msg.lower() or '401' in error_msg:
        print(f"\nLoi xac thuc HuggingFace!")
        print(f"Vui long chay: huggingface-cli login")
        print(f"Hoac set token: export HF_TOKEN=<your_token>")
    else:
        print(f"\nLoi upload: {e}")
        raise

It seems you are trying to upload a large folder at once. This might take some time and then fail if the folder is too large. For such cases, it is recommended to upload in smaller batches or to use `HfApi().upload_large_folder(...)`/`hf upload-large-folder` instead. For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/upload#upload-a-large-folder.


Uploading /home/sakana/Code/ManifoldsTeam/woodbench/data/task-A len ny0g507h3p/wood-task-A...
Luu y: Qua trinh upload co the mat vai phut tuy kich thuoc dataset.

Loi upload: (Request ID: Root=1-6a0763d1-4518aeb02abead636aa7acee;7cb75ba5-30d2-4704-a0f7-213332467a3d)

403 Forbidden: Forbidden: you must use a write token to upload to a repository..
Cannot access content at: https://huggingface.co/api/datasets/ny0g507h3p/wood-task-A/preupload/main.
Make sure your token has the correct permissions.


HfHubHTTPError: (Request ID: Root=1-6a0763d1-4518aeb02abead636aa7acee;7cb75ba5-30d2-4704-a0f7-213332467a3d)

403 Forbidden: Forbidden: you must use a write token to upload to a repository..
Cannot access content at: https://huggingface.co/api/datasets/ny0g507h3p/wood-task-A/preupload/main.
Make sure your token has the correct permissions.